# Persistent Homology for Drug-Target Interaction

This notebook builds the TDA pipeline used in **TopoSurface-DTI** from first principles.

| Section | What you will build |
|---|---|
| 1 | Why topology for molecules? |
| 2 | Simplicial complexes by hand |
| 3 | Vietoris-Rips filtration, step by step |
| 4 | Boundary operators and homology groups |
| 5 | The reduction algorithm (lowest-1 pivot) |
| 6 | What Ripser actually runs (and why it is 1000× faster) |
| 7 | Persistence diagrams and barcodes |
| 8 | Persistence images — vectorizing topology |
| 9 | End-to-end: drug vs. protein pocket |


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.collections import LineCollection, PolyCollection
from mpl_toolkits.mplot3d import Axes3D
import torch

plt.rcParams.update({'font.size': 11, 'figure.dpi': 110})
print('Setup complete.')

---
## 1  Why topology for molecules?

**Binding affinity depends on shape — not just coordinates.**

* A benzene ring is topologically different from a chain: it encloses a 1-cycle (loop).
* A protein binding pocket is a **void** (2-cycle) that a drug docks into.
* Two conformations of the same molecule can have the same bond graph but different numbers of rings.

Standard GNNs encode each atom's local neighbourhood but miss **global** shape features like
how many independent rings exist or how a cavity is shaped.  
Persistent homology computes these features **directly from the 3-D point cloud**, without needing a predefined graph.

The key invariants:
- **H₀** — connected components (how many disconnected clusters?)
- **H₁** — independent loops / rings
- **H₂** — enclosed voids / cavities

In [ ]:
# Demonstrate the difference between a ring and a chain with the same number of points
n = 8
angles = np.linspace(0, 2*np.pi, n, endpoint=False)
ring  = np.stack([np.cos(angles), np.sin(angles)], axis=1)
chain = np.stack([np.linspace(-1, 1, n), np.zeros(n)], axis=1)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
for ax, pts, title, color in zip(axes, [ring, chain],
                                  ['Ring: has an H₁ loop', 'Chain: no loop'],
                                  ['#e07b54', '#5b8db8']):
    ax.scatter(pts[:,0], pts[:,1], s=120, zorder=3, color=color)
    # connect nearby points
    for i in range(len(pts)):
        for j in range(i+1, len(pts)):
            if np.linalg.norm(pts[i]-pts[j]) < 0.9:
                ax.plot([pts[i,0],pts[j,0]], [pts[i,1],pts[j,1]], 'k-', lw=1.2, alpha=0.5)
    ax.set_title(title, fontweight='bold')
    ax.set_aspect('equal'); ax.axis('off')
plt.tight_layout()
plt.show()
print('Both have 8 points and ~8 edges, but only the ring encloses a cycle.')

---
## 2  Simplicial Complexes

A **simplicial complex** K is a collection of simplices closed under taking faces.

| Dimension | Name | Example |
|---|---|---|
| 0 | vertex | a point |
| 1 | edge | a line segment between 2 vertices |
| 2 | triangle | a filled triangle (3 vertices + 3 edges + interior) |
| 3 | tetrahedron | 4 vertices + 6 edges + 4 triangles + interior |

**Closure rule:** if a triangle [a,b,c] ∈ K then all its edges [a,b],[a,c],[b,c] and vertices [a],[b],[c] must also be in K.

In [ ]:
def draw_complex(ax, vertices, edges, triangles, title=''):
    """Draw a 2D simplicial complex with coloured layers."""
    # triangles (filled)
    for tri in triangles:
        poly = plt.Polygon(vertices[list(tri)], closed=True,
                           facecolor='#a8d8ea', edgecolor='none', alpha=0.5)
        ax.add_patch(poly)
    # edges
    for u, v in edges:
        ax.plot([vertices[u,0], vertices[v,0]],
                [vertices[u,1], vertices[v,1]], 'k-', lw=2)
    # vertices
    ax.scatter(vertices[:,0], vertices[:,1], s=200, zorder=5,
               color='#e07b54', edgecolors='k', linewidths=1.5)
    for i, (x, y) in enumerate(vertices):
        ax.text(x, y+0.12, str(i), ha='center', fontsize=10, fontweight='bold')
    ax.set_title(title, fontweight='bold')
    ax.set_aspect('equal'); ax.axis('off')

# Define a small example complex
V = np.array([[0,0],[1,0],[0.5,0.9],[1.5,0.9],[2,0]], dtype=float)

fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))

# Just vertices
draw_complex(axes[0], V, [], [], 'Vertices only (K₀)')

# Vertices + edges
E = [(0,1),(1,2),(0,2),(2,3),(3,4),(1,3)]
draw_complex(axes[1], V, E, [], 'Add edges (K₁)')

# Add filled triangle [0,1,2]
draw_complex(axes[2], V, E, [(0,1,2)], 'Fill triangle [0,1,2] (K₂)')

plt.tight_layout()
plt.show()

print('Triangle [0,1,2] kills the loop 0→1→2→0 (that H₁ class becomes trivial).')
print('The loop 1→2→3→1 still survives as an independent H₁ generator.')

---
## 3  Vietoris-Rips Filtration

Given a point cloud P and a radius ε, the **Vietoris-Rips complex** VR(P, ε) contains every simplex
whose vertices all lie within pairwise distance ε.

A **filtration** is the nested family:
$$\emptyset = K_0 \subseteq K_{\varepsilon_1} \subseteq K_{\varepsilon_2} \subseteq \cdots \subseteq K_{\varepsilon_\text{max}}$$

As ε grows:
1. At ε = 0: only isolated vertices.
2. When ε reaches dist(p, q): edge [p,q] appears.
3. When ε reaches max(dist(a,b), dist(a,c), dist(b,c)): triangle [a,b,c] appears.

A topological feature **is born** when it first appears and **dies** when it is filled in.

In [ ]:
np.random.seed(42)
# Six points arranged roughly in a circle (like a ring in a molecule)
angles_pts = np.linspace(0, 2*np.pi, 6, endpoint=False) + 0.15
pts = np.stack([np.cos(angles_pts)*1.0 + np.random.randn(6)*0.12,
                np.sin(angles_pts)*1.0 + np.random.randn(6)*0.12], axis=1)

# pairwise distances
N = len(pts)
D = np.sqrt(((pts[:,None,:] - pts[None,:,:])**2).sum(-1))

eps_vals = [0.0, 0.8, 1.2, 1.8, 2.5]
fig, axes = plt.subplots(1, len(eps_vals), figsize=(15, 3.2))

for ax, eps in zip(axes, eps_vals):
    # draw circles
    for p in pts:
        circ = plt.Circle(p, eps/2, color='#5b8db8', alpha=0.08)
        ax.add_patch(circ)
    # draw triangles first
    for i in range(N):
        for j in range(i+1, N):
            for k in range(j+1, N):
                if max(D[i,j], D[i,k], D[j,k]) <= eps:
                    tri = plt.Polygon(pts[[i,j,k]], closed=True,
                                      facecolor='#a8d8ea', edgecolor='none', alpha=0.6)
                    ax.add_patch(tri)
    # draw edges
    for i in range(N):
        for j in range(i+1, N):
            if D[i,j] <= eps:
                ax.plot([pts[i,0],pts[j,0]], [pts[i,1],pts[j,1]], 'k-', lw=1.8)
    # draw points
    ax.scatter(pts[:,0], pts[:,1], s=120, zorder=5, color='#e07b54', edgecolors='k')
    n_edges = sum(1 for i in range(N) for j in range(i+1,N) if D[i,j]<=eps)
    n_tri   = sum(1 for i in range(N) for j in range(i+1,N) for k in range(j+1,N)
                  if max(D[i,j],D[i,k],D[j,k])<=eps)
    ax.set_title(f'ε = {eps:.1f}\n{n_edges}E  {n_tri}T', fontsize=9, fontweight='bold')
    ax.set_xlim(-1.8, 1.8); ax.set_ylim(-1.8, 1.8)
    ax.set_aspect('equal'); ax.axis('off')

plt.suptitle('Vietoris-Rips filtration on 6 ring-like points', fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('Notice: a loop (H₁) appears around ε≈0.8 and is killed when the interior fills at ε≈2.5.')
print('The birth-death pair (0.8, 2.5) records the ring.')

---
## 4  Boundary Operators and Homology

**Boundary operator** ∂ₙ maps n-simplices to their (n−1)-dimensional boundaries:

$$\partial_1([u,v]) = [v] - [u]$$
$$\partial_2([u,v,w]) = [v,w] - [u,w] + [u,v]$$

We work over **GF(2)** (coefficients mod 2) so −1 ≡ 1.  
Then ∂ₙ is a matrix of 0s and 1s.

**Key chain of maps:** 
$$\cdots \xrightarrow{\partial_3} C_2 \xrightarrow{\partial_2} C_1 \xrightarrow{\partial_1} C_0 \xrightarrow{\partial_0} 0$$

**Fundamental property:** $\partial_n \circ \partial_{n+1} = 0$  (the boundary of a boundary is empty)

This means:
- $B_n = \text{im}(\partial_{n+1})$ — boundaries (trivial cycles)
- $Z_n = \ker(\partial_n)$ — cycles
- $H_n = Z_n / B_n$ — **homology** = non-trivial cycles

In [ ]:
# Build explicit boundary matrices for a small triangle complex
# Vertices: 0, 1, 2, 3
# Edges: e0=[0,1], e1=[0,2], e2=[1,2], e3=[2,3]
# Triangles: t0=[0,1,2]

vertices  = [0, 1, 2, 3]
edges     = [(0,1), (0,2), (1,2), (2,3)]  # indexed 0,1,2,3
triangles = [(0,1,2)]                       # indexed 0

# ∂₁: edges → vertices   shape (n_vert, n_edge)
B1 = np.zeros((len(vertices), len(edges)), dtype=int)
for j, (u, v) in enumerate(edges):
    B1[u, j] = 1
    B1[v, j] = 1

# ∂₂: triangles → edges  shape (n_edge, n_tri)
edge_idx = {e: i for i, e in enumerate(edges)}
B2 = np.zeros((len(edges), len(triangles)), dtype=int)
for j, (a, b, c) in enumerate(triangles):
    for u, v in [(a,b), (a,c), (b,c)]:
        key = (min(u,v), max(u,v))
        B2[edge_idx[key], j] = 1

print('∂₁  (vertices × edges)  — columns are edges:')
edge_labels = [f'[{u},{v}]' for u,v in edges]
vert_labels = [f'v{i}' for i in vertices]
header = '       ' + '  '.join(f'{l:6s}' for l in edge_labels)
print(header)
for i, row in enumerate(B1):
    print(f'  {vert_labels[i]:4s} ' + '  '.join(f'{x:6d}' for x in row))

print()
print('∂₂  (edges × triangles)  — columns are triangles:')
tri_labels = [f'[{a},{b},{c}]' for a,b,c in triangles]
print('       ' + '  '.join(f'{l:10s}' for l in tri_labels))
for i, row in enumerate(B2):
    print(f'  {edge_labels[i]:7s} ' + '  '.join(f'{x:10d}' for x in row))

print()
# Verify ∂₁ ∘ ∂₂ = 0 (mod 2)
product = (B1 @ B2) % 2
print(f'∂₁ ∘ ∂₂ (mod 2) = {product.tolist()}   ← should be all zeros')

In [ ]:
# Visualise what ∂₁ means: the boundary of edge [1,2] is {1, 2}
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
V2 = np.array([[0,0],[1,0],[0.5,0.9],[1.5,0.5]], dtype=float)

examples = [
    ('∂₁([1,2]) = {1, 2}', [(1,2)], [], [1,2]),
    ('∂₂([0,1,2]) = [0,1]⊕[0,2]⊕[1,2]', [(0,1),(0,2),(1,2)], [(0,1,2)], [0,1,2]),
    ('A cycle: 0→1→2→0 \n(in ker ∂₁, not in im ∂₂ ⟹ H₁ generator)', [(0,1),(1,2),(0,2)], [], [0,1,2]),
]

for ax, (title, es, tris, hi_verts) in zip(axes, examples):
    for (a,b,c) in tris:
        poly = plt.Polygon(V2[[a,b,c]], closed=True,
                           facecolor='#a8d8ea', edgecolor='none', alpha=0.5)
        ax.add_patch(poly)
    all_edges = set()
    for (a,b,c) in tris:
        all_edges |= {(a,b),(a,c),(b,c)}
    for u,v in [(0,1),(0,2),(0,3),(1,2),(2,3)]:
        c = '#333' if (u,v) not in [(eu,ev) for eu,ev in es] else '#e07b54'
        lw = 1 if (u,v) not in [(eu,ev) for eu,ev in es] else 3
        ax.plot([V2[u,0],V2[v,0]], [V2[u,1],V2[v,1]], color=c, lw=lw)
    colors = ['#e07b54' if i in hi_verts else '#aaa' for i in range(4)]
    ax.scatter(V2[:,0], V2[:,1], s=150, c=colors, zorder=5, edgecolors='k')
    for i, (x,y) in enumerate(V2):
        ax.text(x, y+0.1, str(i), ha='center', fontsize=10)
    ax.set_title(title, fontsize=9, fontweight='bold')
    ax.set_aspect('equal'); ax.axis('off')

plt.tight_layout()
plt.show()

---
## 5  The Reduction Algorithm (Lowest-1 Pivot)

**Goal:** given the boundary matrix B (columns = simplices ordered by filtration time), 
find all persistence pairs (σ, τ) where σ is born and τ kills it.

**Algorithm** (over GF(2)):
```
for each column j (left to right):
    while low(j) == low(k) for some k < j:
        column_j = column_j XOR column_k
    if column_j ≠ 0:
        record pair (low(j), j)    # low(j) is born, j kills it
```

- `low(j)` = index of the **lowest non-zero row** in column j
- A column that reduces to zero → the simplex creates a new cycle (it is born)
- A column with a pivot at row r → simplex j kills the cycle born at simplex r

In [ ]:
def reduce_boundary_matrix(B, verbose=True):
    """
    Standard column reduction over GF(2).
    Returns:
        pairs : list of (birth_simplex_idx, death_simplex_idx)
        essential : list of simplex indices with no pair (essential = infinite bar)
    """
    B = B.copy() % 2
    n_cols = B.shape[1]
    low = {}  # pivot_row -> column_index
    pairs = []

    if verbose:
        print(f'Initial boundary matrix ({B.shape[0]} rows × {B.shape[1]} cols):')
        print(B)
        print()

    for j in range(n_cols):
        nz = np.where(B[:, j] != 0)[0]
        while len(nz) > 0:
            piv = int(nz[-1])
            if piv not in low:
                low[piv] = j
                break
            # XOR with the column that owns this pivot
            B[:, j] = (B[:, j] + B[:, low[piv]]) % 2
            nz = np.where(B[:, j] != 0)[0]

        nz = np.where(B[:, j] != 0)[0]
        if len(nz) > 0:
            piv = int(nz[-1])
            pairs.append((piv, j))
            if verbose:
                print(f'  col {j} has pivot at row {piv}  →  pair ({piv}, {j})')

    paired = {s for p in pairs for s in p}
    essential = [j for j in range(n_cols) if j not in paired
                 and np.all(B[:, j] == 0)]

    if verbose:
        print(f'\nReduced matrix:')
        print(B)

    return pairs, essential


# ── Worked example: 4-point ring ─────────────────────────────────────────────
# Points: 0,1,2,3 arranged as a square
# Filtration order: vertices first, then edges by length
# v0(0) v1(0) v2(0) v3(0) | e01(1) e12(1) e23(1) e03(1) | e02(√2) e13(√2)
# Simplex indices:  0  1   2   3      4      5      6      7        8      9

# B₁: edges (cols 4-9) vs vertices (rows 0-3)
B1_ring = np.array([
#   e01 e12 e23 e03 e02 e13
    [1,  0,  0,  1,  1,  0],  # v0
    [1,  1,  0,  0,  0,  1],  # v1
    [0,  1,  1,  0,  1,  0],  # v2
    [0,  0,  1,  1,  0,  1],  # v3
], dtype=int)

print('=== H₀ reduction (∂₁, finds when components merge) ===\n')
pairs_h0, ess_h0 = reduce_boundary_matrix(B1_ring)
print(f'\nPairs (killed_vertex, killing_edge): {pairs_h0}')
print(f'Essential (never killed): {ess_h0}  ← one H₀ = one final component')

In [ ]:
# Now H₁: add the boundary matrix ∂₂ for any triangles
# For the 4-point square, there are no triangles within short edges only,
# so the loop 0→1→2→3→0 is an essential H₁ class.
# Let's use a 3-point triangle to see a loop get killed.

# 3 points: 0,1,2.  Edges: [0,1] at ε=1, [0,2] at ε=1, [1,2] at ε=1.2
# Triangle [0,1,2] appears at ε=1.2 (max edge)
# Simplex order: v0 v1 v2 | e01 e02 | e12 | t012
#                 0   1   2     3    4      5      6

# ∂₂: columns = triangles, rows = edges (e01=0, e02=1, e12=2)
B2_tri = np.array([
#   t012
    [1],   # e01 appears in boundary of t012
    [1],   # e02
    [1],   # e12
], dtype=int)

# B1 for this complex (edges vs vertices)
B1_tri = np.array([
#   e01 e02 e12
    [1,  1,  0],   # v0
    [1,  0,  1],   # v1
    [0,  1,  1],   # v2
], dtype=int)

print('=== H₁ reduction (∂₂, finds when loops are filled by triangles) ===\n')
print('Columns = triangles; rows = edges')
pairs_h1, ess_h1 = reduce_boundary_matrix(B2_tri)

print(f'\nPairs: {pairs_h1}')
if pairs_h1:
    row, col = pairs_h1[0]
    print(f'  edge {row} (e12, born at ε=1.2) killed by triangle {col} (born at ε=1.2)')
    print(f'  → persistence = 1.2 - 1.2 = 0  (triangle immediately kills the loop it closes)')
else:
    print('  No H₁ pairs — the loop is essential (never killed within max_edge_len)')

---
## 6  What Ripser Actually Runs

Standard reduction is O(n³) per matrix — too slow for proteins with 200 Cα atoms.
Ripser uses three key ideas:

### 6.1  Coboundary (Dual) Representation

Instead of working with ∂ₙ (boundary), Ripser works with its **transpose** δₙ = ∂ₙᵀ (coboundary).  
This is much **sparser**: each triangle touches only 3 edges, so δ₁ has 3 ones per column vs potentially many more in the boundary direction.

### 6.2  Apparent Pairs

A pair (σ, τ) is **apparent** if:
- τ is the cofacet of σ with the smallest filtration value
- σ is the facet of τ with the largest filtration value

These pairs can be found in O(1) per simplex without any matrix reduction.  
Empirically, **>95%** of all pairs are apparent for molecular point clouds.

### 6.3  Clearing Lemma

If column j has already been identified as a "killer" (it has a pivot), then the column it kills must reduce to zero.  
**We can skip reducing that zero column entirely.**  
This clears roughly half the work.

In [ ]:
# Simulate the apparent pairs speedup
import time

def count_apparent_pairs(pts, max_r):
    """Count how many edge-triangle pairs are 'apparent' — no reduction needed."""
    N = len(pts)
    D = np.sqrt(((pts[:,None,:]-pts[None,:,:])**2).sum(-1))
    
    edges = [(D[i,j], i, j) for i in range(N) for j in range(i+1,N) if D[i,j]<=max_r]
    edges.sort()
    
    apparent = 0
    total_tri = 0
    for i in range(N):
        for j in range(i+1,N):
            if D[i,j] > max_r: continue
            for k in range(j+1,N):
                if D[i,k]>max_r or D[j,k]>max_r: continue
                total_tri += 1
                # The triangle's "youngest" edge is the one with max distance
                dij,dik,djk = D[i,j],D[i,k],D[j,k]
                max_d = max(dij,dik,djk)
                # The "oldest" face of the triangle is the edge with max_d
                # Apparent pair if that edge's youngest cofacet is THIS triangle
                # (simplified check: the edge's younger neighbor has smaller birth)
                apparent += 1  # simplified: most are apparent
    
    return total_tri, apparent

# Time comparison
rng = np.random.default_rng(7)
print('Comparing Ripser vs scratch fallback on growing point clouds:\n')
print(f'{"N":>5}  {"Ripser (ms)":>12}  {"Scratch (ms)":>13}  {"Speedup":>8}')
print('-'*50)

try:
    from ripser import ripser as _ripser
    import sys
    sys.path.insert(0, '..')
    from data.tda_features import _compute_tda_scratch
    
    for n in [20, 40, 80]:
        pts = rng.random((n, 3)) * 10.0
        
        t0 = time.perf_counter()
        for _ in range(5): _ripser(pts, maxdim=1, thresh=8.0)
        t_ripser = (time.perf_counter()-t0)/5*1000
        
        t0 = time.perf_counter()
        _compute_tda_scratch(pts, 8.0, 20, n, 0)
        t_scratch = (time.perf_counter()-t0)*1000
        
        print(f'{n:>5}  {t_ripser:>12.2f}  {t_scratch:>13.2f}  {t_scratch/t_ripser:>8.1f}×')
except ImportError:
    print('  (ripser not installed — install with: pip install ripser)')
    print('  Expected speedup: 100-1000× for N=20-200')

---
## 7  Persistence Diagrams and Barcodes

After reduction, each homological feature is a pair **(birth ε, death ε)**:

- **Barcode**: horizontal bars from birth to death on an ε-axis
- **Persistence diagram**: each pair plotted as a point (birth, death) above the diagonal
- **Persistence** = death − birth: long bars = robust topological features

Infinite bars (never killed) represent **essential classes** — e.g., the one component that remains forever.

In [ ]:
sys.path.insert(0, '..')
from data.tda_features import compute_tda_features
from data.molecule_graph import synthetic_drug_graph
from data.pocket_mesh import synthetic_pocket_graph

# Compute TDA for a synthetic drug molecule
drug = synthetic_drug_graph(n_atoms=24, seed=3)
tda_drug = compute_tda_features(drug['pos'].numpy(), max_edge_len=8.0)

h0 = tda_drug['h0_pairs']
h1 = tda_drug['h1_pairs']

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# ── Barcode H₀ ──────────────────────────────────────────────────────────────
ax = axes[0]
max_val = 8.0
for k, (b, d) in enumerate(sorted(h0, key=lambda x: x[0])):
    d_clip = min(d, max_val)
    ax.plot([b, d_clip], [k, k], lw=3,
            color='#5b8db8' if d < max_val*1.4 else '#e07b54')
ax.axvline(max_val, ls='--', color='gray', lw=1, label='max_edge_len')
ax.set_xlabel('ε (Å)'); ax.set_ylabel('bar index')
ax.set_title('H₀ Barcode\n(connected components)', fontweight='bold')
ax.legend(fontsize=8)

# ── Barcode H₁ ──────────────────────────────────────────────────────────────
ax = axes[1]
for k, (b, d) in enumerate(sorted(h1, key=lambda x: x[1]-x[0], reverse=True)):
    ax.plot([b, d], [k, k], lw=3, color='#e07b54')
ax.set_xlabel('ε (Å)'); ax.set_ylabel('bar index')
ax.set_title('H₁ Barcode\n(independent loops)', fontweight='bold')

# ── Persistence Diagram ──────────────────────────────────────────────────────
ax = axes[2]
lim = max_val * 1.1
ax.plot([0, lim], [0, lim], 'k--', lw=1, alpha=0.4)
if h0:
    b0 = [p[0] for p in h0]; d0 = [min(p[1],max_val) for p in h0]
    ax.scatter(b0, d0, s=60, color='#5b8db8', label='H₀', zorder=3)
if h1:
    b1 = [p[0] for p in h1]; d1 = [p[1] for p in h1]
    ax.scatter(b1, d1, s=60, color='#e07b54', marker='^', label='H₁', zorder=3)
ax.set_xlabel('Birth ε (Å)'); ax.set_ylabel('Death ε (Å)')
ax.set_title('Persistence Diagram', fontweight='bold')
ax.legend()
ax.set_xlim(-0.2, lim); ax.set_ylim(-0.2, lim)

plt.suptitle(f'Drug molecule — {len(h0)} H₀ bars, {len(h1)} H₁ bars',
             fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f'Most persistent H₁ loop: birth={max(h1, key=lambda x:x[1]-x[0])[0]:.2f}Å,'
      f' death={max(h1, key=lambda x:x[1]-x[0])[1]:.2f}Å' if h1 else 'No H₁ loops found')

---
## 8  Persistence Images — Vectorizing Topology

A persistence diagram is a **set of points** — not a fixed-size vector.  
Neural networks need fixed-size inputs.  

**Persistence image** (Adams et al. 2017):
1. Transform diagram to **(birth, persistence=death−birth)** space
2. Weight each point by its persistence (long-lived features matter more)
3. Sum weighted Gaussians on a regular grid
4. Flatten to a 1D vector

$$\text{PI}[i,j] = \sum_{(b,d) \in \text{dgm}} (d-b) \cdot \exp\!\left(-\frac{(g_i^b - b)^2 + (g_j^p - (d-b))^2}{2\sigma^2}\right)$$

In this project: **20×20 grid → 400 floats per diagram**, H₀ + H₁ concatenated → **800-dim vector**.

In [ ]:
from data.tda_features import persistence_image

def visualize_persistence_image(pairs, max_val, resolution=20, title='', ax=None):
    """Show the persistence image alongside the persistence diagram."""
    img = persistence_image(pairs, resolution=resolution, max_val=max_val)
    img_2d = img.reshape(resolution, resolution)
    
    if ax is None:
        fig, ax = plt.subplots(figsize=(4,4))
    
    im = ax.imshow(img_2d.T, origin='lower', extent=[0, max_val, 0, max_val],
                   cmap='YlOrRd', aspect='auto')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    
    # Overlay diagram points
    if pairs:
        births = [b for b,d in pairs]
        persts = [d-b for b,d in pairs]
        ax.scatter(births, persts, s=30, color='navy', zorder=5, alpha=0.7)
    
    ax.set_xlabel('Birth (Å)'); ax.set_ylabel('Persistence = death−birth (Å)')
    ax.set_title(title, fontweight='bold')
    return img


fig, axes = plt.subplots(1, 4, figsize=(16, 4))

# Drug H₀ and H₁
visualize_persistence_image(h0, 8.0, title='Drug H₀ image\n(components)', ax=axes[0])
visualize_persistence_image(h1, 8.0, title='Drug H₁ image\n(loops)', ax=axes[1])

# Pocket H₀ and H₁ (larger max_edge_len because proteins are bigger)
pocket = synthetic_pocket_graph(n_residues=30, seed=1)
tda_pocket = compute_tda_features(pocket['pos'].numpy(), max_edge_len=16.0)
visualize_persistence_image(tda_pocket['h0_pairs'], 16.0,
                             title='Pocket H₀ image\n(components)', ax=axes[2])
visualize_persistence_image(tda_pocket['h1_pairs'], 16.0,
                             title='Pocket H₁ image\n(loops/voids)', ax=axes[3])

plt.suptitle('Persistence images (20×20 grid, weighted Gaussians)', fontweight='bold')
plt.tight_layout()
plt.show()

print('Each image is a 400-float vector. Concatenating H₀+H₁ gives 800-dim TDA descriptor.')
print('Drug and pocket both produce 800-dim → 1600 total fed into FusionModule.')

In [ ]:
# Demonstrate sensitivity: same topology, different sigma -> different smoothing
fig, axes = plt.subplots(1, 4, figsize=(16, 3.8))

for ax, sigma_mult in zip(axes, [0.5, 1.0, 2.0, 4.0]):
    max_val = 8.0
    resolution = 20
    sigma = max_val / resolution * sigma_mult  # custom sigma
    
    pairs = h1
    if not pairs:
        ax.text(0.5, 0.5, 'No H₁ pairs', ha='center', transform=ax.transAxes)
        continue
    
    births   = np.array([b for b,d in pairs])
    persts   = np.array([d-b for b,d in pairs])
    gb = np.linspace(0, max_val, resolution)
    gp = np.linspace(0, max_val, resolution)
    img = np.zeros((resolution, resolution))
    for b, p in zip(births, persts):
        if p < 1e-8: continue
        img += p * np.outer(
            np.exp(-((gb-b)**2)/(2*sigma**2)),
            np.exp(-((gp-p)**2)/(2*sigma**2))
        )
    if img.max() > 0: img /= img.max()
    
    ax.imshow(img.T, origin='lower', extent=[0,max_val,0,max_val],
              cmap='YlOrRd', aspect='auto')
    ax.scatter(births, persts, s=25, color='navy', zorder=5, alpha=0.7)
    ax.set_title(f'σ = {sigma_mult}× default\n(σ={sigma:.2f}Å)', fontweight='bold')
    ax.set_xlabel('Birth'); ax.set_ylabel('Persistence')

plt.suptitle('Effect of σ on persistence image (H₁ of drug)', fontweight='bold')
plt.tight_layout()
plt.show()
print('Project default: sigma = max_val/resolution * 2.0 — balances locality and smoothness.')

---
## 9  End-to-End: Drug vs. Protein Pocket in TopoSurface-DTI

How the 800-dim TDA vector flows through the model:

```
drug_pos (N×3)  ──→  compute_tda_features(max_edge_len=8Å)  ──→  h0_img + h1_img  ──→  (800,)
                                                                                             ↓
pocket_pos (V×3) ──→  compute_tda_features(max_edge_len=16Å) ──→  h0_img + h1_img  ──→  (800,)
                                                                                             ↓
                                                   concatenate drug_tda + pocket_tda  →  (1600,)
                                                                                             ↓
                                          FusionModule: cross-attention output + tda_inject MLP
```

**Why different max_edge_len?**  
Drug atoms are ~1–3Å apart; protein Cα atoms in pockets are ~5–15Å apart.  
Using the same filtration radius for both would miss pocket-scale topology (8Å is too short)
or swamp drug features with noise (16Å connects almost everything in a small drug).

In [ ]:
from data.tda_features import tda_to_tensor

drug   = synthetic_drug_graph(n_atoms=24, seed=0)
pocket = synthetic_pocket_graph(n_residues=30, seed=0)

tda_drug_dict   = compute_tda_features(drug['pos'].numpy(),   max_edge_len=8.0)
tda_pocket_dict = compute_tda_features(pocket['pos'].numpy(), max_edge_len=16.0)

drug_vec   = tda_to_tensor(tda_drug_dict)
pocket_vec = tda_to_tensor(tda_pocket_dict)
combined   = torch.cat([drug_vec, pocket_vec])

print(f'Drug TDA vector shape:   {drug_vec.shape}   (H₀: 400 + H₁: 400)')
print(f'Pocket TDA vector shape: {pocket_vec.shape}   (H₀: 400 + H₁: 400)')
print(f'Combined TDA input:      {combined.shape}  (fed into FusionModule)')
print()

fig, axes = plt.subplots(2, 2, figsize=(12, 7))
labels = [
    ('Drug H₀\n(component merging, 8Å scale)', drug_vec[:400]),
    ('Drug H₁\n(ring detection, 8Å scale)',    drug_vec[400:]),
    ('Pocket H₀\n(cluster structure, 16Å)',    pocket_vec[:400]),
    ('Pocket H₁\n(loops/cavities, 16Å)',       pocket_vec[400:]),
]
for ax, (title, v) in zip(axes.flat, labels):
    im = ax.imshow(v.reshape(20,20).numpy().T, origin='lower', cmap='YlOrRd', aspect='auto')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    ax.set_title(title, fontweight='bold', fontsize=10)
    ax.set_xlabel('Birth axis'); ax.set_ylabel('Persistence axis')

plt.suptitle('All four TDA images fed into TopoSurface-DTI', fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Full forward pass to confirm dimensions
from models.toposurface_dti import TopoSurfaceDTI

model = TopoSurfaceDTI()
pred = model(
    drug_x=drug['x'], drug_pos=drug['pos'], drug_edge=drug['edge_index'],
    pocket_x=pocket['x'], pocket_edge=pocket['edge_index'],
    pocket_angles=pocket['angles'], pocket_trans=pocket['transporters'],
    drug_tda=drug_vec, pocket_tda=pocket_vec,
)

params = model.count_parameters()
print('Parameter counts:')
for k, v in params.items():
    print(f'  {k:25s}: {v:>7,}')

print(f'\nFinal pKd prediction: {pred.item():.4f}')
print('\nSummary: 800 TDA dims per molecule capture global topology.')
print('Combined with local GEM-CNN pocket features, this gives the model')
print('both fine-grained surface geometry AND coarse global shape.')

---
## Summary

| Concept | What it does in the project |
|---|---|
| Vietoris-Rips filtration | Grows a complex from pairwise distances — no predefined graph needed |
| H₀ persistence | Detects when atoms cluster; long bars = isolated fragments |
| H₁ persistence | Detects rings and tunnels; critical for aromatic drug scaffolds |
| Boundary matrix reduction | The reference algorithm — correct but O(n³) |
| Ripser apparent pairs | Handles >95% of pairs without any matrix operations |
| Clearing lemma | Skips zero columns — halves the remaining work |
| Persistence image | Fixed-size 400-float vector per diagram (20×20 weighted Gaussians) |
| max_edge_len | 8Å for drugs, 16Å for proteins — matched to physical scale |
| Combined TDA vector | 1600-dim input to FusionModule alongside GEM-CNN graph embeddings |

**Key insight:** GNN features encode *local* neighbourhood chemistry. TDA features encode *global* molecular shape — rings, cavities, and connectivity patterns that span the entire molecule. Together they give the model complementary views of the binding interaction.